In [ ]:
from datasets import load_dataset

In [ ]:
data = load_dataset("lmms-lab/LiveBench", "2024-09", split="test")

In [ ]:
from tqdm import tqdm

subsets = {}
for item in tqdm(data, total=len(data)):
    subset = eval(item["website"])["subject"]
    if subset not in subsets:
        subsets[subset] = 0
    subsets[subset] += 1

In [ ]:
subsets

In [ ]:
sum(subsets.values()) / len(subsets)

In [ ]:
df = data.to_pandas()

In [ ]:
def update(row):
    if row["subtask"] == "Evaluative Questions":
        row["subtask"] = "Divergent Thinking"

In [ ]:
df.apply(update, axis=1)

In [ ]:
df = df[df["score"] > 6]

In [ ]:
len(df)

In [ ]:
from tqdm import tqdm

subsets = {}
for idx, item in tqdm(df.iterrows(), total=len(df)):
    subset = eval(item["website"])["subject"]
    if subset not in subsets:
        subsets[subset] = 0
    subsets[subset] += 1

In [ ]:
final_data = {}
for idx, item in tqdm(df.iterrows(), total=len(df)):
    subtask = item["subtask"]
    if subtask == "Evaluative Questions":
        subtask = "Divergent Thinking"
        item["subtask"] = subtask
    if subtask not in final_data:
        final_data[subtask] = []
    final_data[subtask].append(item)

In [ ]:
del final_data["Further Insights"]

In [ ]:
final_data.keys()

In [ ]:
from random import shuffle


for key, value in final_data.items():
    shuffle(value)
    value = sorted(value, key=lambda x: x["score"])
    value = list(reversed(value))[:50]
    final_data[key] = value

In [ ]:
import pandas as pd

final_df = pd.concat([pd.DataFrame(value) for value in final_data.values()])

In [ ]:
subsets

In [ ]:
from tqdm import tqdm

subsets = {}
for idx, item in tqdm(final_df.iterrows(), total=len(final_df)):
    subset = item["subtask"]
    if subset not in subsets:
        subsets[subset] = 0
    subsets[subset] += 1

In [ ]:
len(final_df)

In [ ]:
subsets

In [ ]:
df = df[df["subtask"] != "Further Insights"]

In [ ]:
from datasets import Dataset

new_data = Dataset.from_pandas(final_df, preserve_index=False, features=data.features)

In [ ]:
new_data[0]

In [ ]:
new_data

In [ ]:
new_data.push_to_hub("lmms-lab/LiveBench", "2024-09", split="test")

In [ ]:
new_data = load_dataset("lmms-lab/LiveBench", "2024-09", split="test")
df = new_data.to_pandas()

In [ ]:
def get_subset(row):
    return eval(row)["subject"]


df["website"] = df["website"].apply(get_subset)

In [ ]:
df.iloc[0]["website"]

In [ ]:
new_data = Dataset.from_pandas(df, preserve_index=False, features=new_data.features)
new_data.push_to_hub("lmms-lab/LiveBench", "2024-09", split="test")

In [ ]:
from PIL import Image
import io


def get_data():
    for index, item in df.iterrows():
        new_item = item.to_dict()
        new_item["subset"] = eval(item["website"])["subject"]
        del new_item["website"]
        new_item["id"] = index
        image_bytes = new_item["images"][0]["bytes"]
        image = Image.open(io.BytesIO(image_bytes))
        new_item["images"] = [image]
        yield new_item


dataset = Dataset.from_generator(get_data)

In [ ]:
dataset.push_to_hub("lmms-lab/LiveBench", "2024-09", split="test")